# 4) Functions Challenge — Capstone Exercises

**Goals:** design small, reusable functions; pass functions around; safe defaults; error handling; docstrings & tests; gluing pieces into a mini pipeline.

You can solve these by composing helpers you wrote earlier.

### A) Mini ETL: transactions → daily totals

Input: CSV lines like:

```
2025-08-01,alice,+10.50
2025-08-01,alice,-2
2025-08-02,bob,+5
bad,line
```

Produce: `{"2025-08-01": {"alice": 8.5}, "2025-08-02": {"bob": 5.0}}`

```python
def parse_line(line):
    """
    Return (date, user, amount_float) or None if invalid.
    Amount may have '+' or '-' sign; spaces allowed.
    """
    ...
def accumulate(lines):
    """
    Use parse_line; return nested dict date->{user: total}.
    """
    ...
lines = ["2025-08-01,alice,+10.50","2025-08-01,alice,-2","2025-08-02,bob,+5","bad,line"]
out = accumulate(lines)
assert out == {"2025-08-01":{"alice":8.5}, "2025-08-02":{"bob":5.0}}
```

### B) Validation + normalization + mapping

```python
def normalize_user(name, email):
    """
    Return dict {'name': 'Title Case', 'email': lower} or raise ValueError.
    """
    ...
def map_users(rows, *, validator=normalize_user):
    """
    rows: list of (name,email). Use validator to normalize each; skip invalid.
    """
    ...
rows = [(" ada lovelace ","ADA@EXAMPLE.COM"), ("bad","noats")]
ok = map_users(rows)
assert ok == [{"name":"Ada Lovelace","email":"ada@example.com"}]
```


In [42]:
def accumulate(lines):
    """
    Use parse_line; return nested dict date->{user: total}.
    """
    dict_acc = {}
    for line in lines:
        date, dct = parse_line(line)
        if date is not None:
            dict_acc[date] = dict_acc.get(date, 0) + 1
    return dict_acc

In [60]:
def parse_line(line):
    """
    Return (date, user, amount_float) or None if invalid.
    Amount may have '+' or '-' sign; spaces allowed.
    """
    lst = line.split(',')
    if len(lst) == 3:
        date = lst[0]
        dct = {lst[1]:float(lst[2].strip().replace('+',''))}
        return date, dct
    else:
        return None, None
    
from collections import defaultdict

def accumulate(lines):
    """
    Use parse_line; return nested dict date->{user: total}.
    """
    dict_acc = defaultdict(lambda: defaultdict(float))
    
    for line in lines:
        date, dct = parse_line(line)
        if date is not None:
            for user, amount in dct.items():
                dict_acc[date][user] += amount
    return {date: dict(users) for date, users in dict_acc.items()}

lines = ["2025-08-01,alice,+10.50","2025-08-01,alice,-2","2025-08-02,bob,+5","bad,line"]
accumulate(lines)

{'2025-08-01': {'alice': 8.5}, '2025-08-02': {'bob': 5.0}}

In [70]:
def normalize_user(name, email):
    """
    Return dict {'name': 'Title Case', 'email': lower} or raise ValueError.
    """
    if '@' not in email:
        raise ValueError
    try:
        return " ".join([i.capitalize() for i in name.strip().split(' ')]), email.strip().lower()
    except:
        raise ValueError   
    
def map_users(rows, validator=normalize_user):
    """
    rows: list of (name,email). Use validator to normalize each; skip invalid.
    """
    lst = []
    for name, email in rows:
        try:
            name, email = validator(name, email)
            lst.append({"name": name, "email": email})
        except:
            pass
    return lst 

rows = [(" ada lovelace ","ADA@EXAMPLE.COM"), ("bad","noats")]

ok = map_users(rows)
print(ok)
assert ok == [{"name":"Ada Lovelace","email":"ada@example.com"}]

[{'name': 'Ada Lovelace', 'email': 'ada@example.com'}]
